# Name That LLM: Traditional Classifier Benchmark

Compares several classical models on the same feature set, then ensembles the best ones.

**Pipeline**
1. Load data and auto-detect the text / label / id columns (check the printout, override in the config cell if wrong)
2. Features: word TF-IDF, character TF-IDF, hand-made stylometric features, SVD embedding
3. Benchmark 7 models on a stratified holdout (balanced accuracy is the headline metric)
4. 5-fold CV of the top models, soft-voting ensemble, class-weight tuning for balanced accuracy
5. Fit on all training data and write `submission.parquet`

Set `MAX_TRAIN` to a small number (e.g. 3000) for a quick smoke test.

In [ ]:
import os, re, time, glob, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.special import softmax

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression, SGDClassifier, RidgeClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (balanced_accuracy_score, accuracy_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

# ---------------- CONFIG ----------------
MAX_TRAIN  = None     # e.g. 3000 for a quick smoke test
TEXT_COL   = None     # None = auto-detect
LABEL_COL  = None
ID_COL     = None
WORD_MAX   = 200_000  # max word n-gram features
CHAR_MAX   = 300_000  # max char n-gram features
SVD_DIMS   = 150
STYLO_W    = 0.5      # weight of stylometric block in the linear feature matrix
HOLDOUT    = 0.2
N_FOLDS    = 5
TOP_K      = 3        # models carried into CV + ensemble
RUN_CV     = True
CANDIDATE_DIRS = [os.environ.get("NTL_DATA_DIR"),
                  "/kaggle/input/competitions/name-that-llm",
                  "/kaggle/input/name-that-llm", "data"]

## 1. Load data

In [ ]:
def find_file(root, stem):
    for ext in ("csv", "parquet"):
        hits = glob.glob(os.path.join(root, "**", f"{stem}*.{ext}"), recursive=True)
        hits = [h for h in hits if "sample" not in os.path.basename(h).lower()] if stem != "sample_submission" else hits
        if hits:
            return sorted(hits)[0]
    return None

def find_root():
    for d in CANDIDATE_DIRS:
        if d and os.path.isdir(d) and find_file(d, "train"):
            return d
    import kagglehub  # local machine: needs `kagglehub.login()` + accepted rules
    return kagglehub.competition_download("name-that-llm")

def read_any(p):
    return pd.read_parquet(p) if p.endswith(".parquet") else pd.read_csv(p)

ROOT = find_root()
train_path, test_path = find_file(ROOT, "train"), find_file(ROOT, "test")
sub_path = find_file(ROOT, "sample_submission")
print("root :", ROOT)
print("files:", os.listdir(ROOT))
train, test = read_any(train_path), read_any(test_path)
print("train", train.shape, "| test", test.shape)
train.head(3)

In [ ]:
def detect_cols(train, test):
    label = LABEL_COL or next((c for c in ["label","labels","model","model_name","target","llm","generator","source","y"]
                               if c in train.columns), None)
    if label is None:
        extra = [c for c in train.columns if c not in test.columns]
        label = extra[0] if extra else None
    obj = [c for c in train.columns if c != label and train[c].dtype == object]
    text = TEXT_COL or next((c for c in ["text","passage","response","answer","content","output","completion"]
                             if c in train.columns), None) \
           or max(obj, key=lambda c: train[c].astype(str).str.len().mean())
    idc = ID_COL or next((c for c in ["id","ID","row_id","idx","index"] if c in test.columns), None)
    return text, label, idc

TEXT_COL, LABEL_COL, ID_COL = detect_cols(train, test)
print(f"TEXT_COL={TEXT_COL!r}  LABEL_COL={LABEL_COL!r}  ID_COL={ID_COL!r}")
assert LABEL_COL, "Could not detect the label column; set LABEL_COL in the config cell."

train = train.dropna(subset=[LABEL_COL]).reset_index(drop=True)
if MAX_TRAIN and len(train) > MAX_TRAIN:
    train = train.groupby(LABEL_COL, group_keys=False).apply(
        lambda g: g.sample(min(len(g), max(1, MAX_TRAIN * len(g) // len(train))), random_state=SEED)
    ).reset_index(drop=True)

le = LabelEncoder()
y = le.fit_transform(train[LABEL_COL])
CLASSES = le.classes_
n_cls, n_train = len(CLASSES), len(train)
print(n_cls, "classes,", n_train, "training rows")

train_text = train[TEXT_COL].fillna("").astype(str)
test_text = test[TEXT_COL].fillna("").astype(str)
lens = train_text.str.len()
print(lens.describe().round(0).to_string())

ax = train[LABEL_COL].value_counts().plot.barh(figsize=(7, max(3, n_cls * .3)), title="Class counts")
plt.show()

## 2. Features

* **Word TF-IDF** (1-2 grams): vocabulary and phrasing habits
* **Char TF-IDF** (1-4 grams, case kept): punctuation, spacing, formatting habits
* **Stylometric block**: lengths, markdown usage, dash/quote styles, list density, etc.
* **SVD** of the word TF-IDF: a dense input for the tree model

The vectorizers are fitted on train + test text together (unsupervised, no labels), so the same matrices serve the holdout, CV and the final fit.

In [ ]:
STYLO_PATTERNS = {
    "em_dash": "—", "en_dash": "–", "bold_md": r"\*\*", "heading_md": r"(?m)^#{1,6} ",
    "bullet": r"(?m)^\s*[-*•] ", "numbered": r"(?m)^\s*\d+[.)] ", "backtick": "`",
    "curly_quote": "[“”]", "curly_apos": "’", "colon": ":", "semicolon": ";", "paren": r"\(",
    "question": r"\?", "exclam": "!", "ellipsis": "…", "pipe": r"\|", "latex": r"\$|\\\(",
    "emoji": "[\U0001F300-\U0001FAFF\u2600-\u27BF]", "dbl_space": "  ",
}

def stylo_features(texts):
    s = pd.Series(list(texts)).fillna("").astype(str)
    n = s.str.len().clip(lower=1)
    words = s.str.split()
    nw = words.str.len().clip(lower=1)
    f = pd.DataFrame(index=s.index)
    f["log_chars"] = np.log1p(n)
    f["log_words"] = np.log1p(nw)
    f["avg_word_len"] = n / nw
    f["log_lines"] = np.log1p(s.str.count("\n"))
    f["log_paras"] = np.log1p(s.str.count("\n\n"))
    f["avg_sent_len"] = nw / s.str.count(r"[.!?](?:\s|$)").clip(lower=1)
    for k, pat in STYLO_PATTERNS.items():
        f[k] = s.str.count(pat) / n * 1000          # rate per 1000 chars
    f["upper_ratio"] = s.str.count(r"[A-Z]") / n
    f["digit_ratio"] = s.str.count(r"\d") / n
    f["nonascii_ratio"] = s.str.count(r"[^\x00-\x7F]") / n
    f["uniq_word_ratio"] = words.apply(lambda w: len(set(w)) / max(len(w), 1))
    f["ends_period"] = s.str.rstrip().str.endswith(".").astype(float)
    f["ends_question"] = s.str.rstrip().str.endswith("?").astype(float)
    return f.astype(np.float32)

t0 = time.time()
all_text = pd.concat([train_text, test_text], ignore_index=True)

word = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=WORD_MAX, sublinear_tf=True, dtype=np.float32)
char = TfidfVectorizer(analyzer="char", ngram_range=(1, 4), min_df=3, max_features=CHAR_MAX,
                       sublinear_tf=True, lowercase=False, dtype=np.float32)
Xw = word.fit_transform(all_text)
Xc = char.fit_transform(all_text)
S = StandardScaler().fit_transform(stylo_features(all_text).values).clip(-6, 6).astype(np.float32)
svd = TruncatedSVD(SVD_DIMS, random_state=SEED).fit_transform(Xw).astype(np.float32)

FEATS = {
    "lin":   sparse.hstack([Xw, Xc, sparse.csr_matrix(S * STYLO_W)]).tocsr(),   # linear models
    "nb":    sparse.hstack([Xw, Xc]).tocsr(),                                    # non-negative only
    "dense": np.hstack([svd, S]),                                                # trees
}
for k, v in FEATS.items():
    print(f"{k:6s}", v.shape)
print(f"features built in {time.time()-t0:.0f}s")

## 3. Benchmark on a stratified holdout

In [ ]:
def get_scores(model, X):
    """Probabilities when available, otherwise a softmax over decision scores (heuristic temperature)."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)
    return softmax(5 * model.decision_function(X), axis=1)

MODELS = {
    "LogReg":        dict(feats="lin",   make=lambda: LogisticRegression(C=10, max_iter=300, class_weight="balanced")),
    "LinearSVC":     dict(feats="lin",   make=lambda: LinearSVC(C=0.3, class_weight="balanced")),
    "SGD (mod.huber)": dict(feats="lin", make=lambda: SGDClassifier(loss="modified_huber", alpha=2e-5, class_weight="balanced",
                                                                    max_iter=40, n_jobs=-1, random_state=SEED)),
    "Ridge":         dict(feats="lin",   make=lambda: RidgeClassifier(alpha=1.0, class_weight="balanced")),
    "ComplementNB":  dict(feats="nb",    make=lambda: ComplementNB(alpha=0.3)),
    "HistGB (SVD+stylo)": dict(feats="dense", sw=True,
                               make=lambda: HistGradientBoostingClassifier(max_iter=300, learning_rate=0.08,
                                                                          early_stopping=True, random_state=SEED)),
}

def fit_model(spec, X, yy):
    m = spec["make"]()
    if spec.get("sw"):
        m.fit(X, yy, sample_weight=compute_sample_weight("balanced", yy))
    else:
        m.fit(X, yy)
    return m

idx_tr, idx_va = train_test_split(np.arange(n_train), test_size=HOLDOUT, stratify=y, random_state=SEED)
rows, holdout_scores = [], {}

for name, spec in MODELS.items():
    X = FEATS[spec["feats"]][:n_train]
    t0 = time.time()
    try:
        m = fit_model(spec, X[idx_tr], y[idx_tr])
        P = get_scores(m, X[idx_va])
    except Exception as e:
        print(f"[skip] {name}: {type(e).__name__}: {e}")
        continue
    pred = P.argmax(1)
    holdout_scores[name] = P
    rows.append(dict(model=name,
                     balanced_acc=balanced_accuracy_score(y[idx_va], pred),
                     accuracy=accuracy_score(y[idx_va], pred),
                     macro_f1=f1_score(y[idx_va], pred, average="macro"),
                     seconds=time.time() - t0))
    print(f"{name:22s} bal_acc={rows[-1]['balanced_acc']:.4f}  ({rows[-1]['seconds']:.0f}s)")

results = pd.DataFrame(rows).sort_values("balanced_acc", ascending=False).reset_index(drop=True)
results.round(4)

In [ ]:
# Quick ensemble on the holdout: mean of the top-K models' probabilities
top_names = results["model"].head(TOP_K).tolist()
ens = np.mean([holdout_scores[n] for n in top_names], axis=0)
print("Top-K:", top_names)
print("Holdout ensemble balanced acc:", round(balanced_accuracy_score(y[idx_va], ens.argmax(1)), 4))

ax = results.set_index("model")[["balanced_acc", "macro_f1"]].plot.barh(figsize=(8, 4), title="Holdout results")
ax.invert_yaxis(); plt.show()

## 4. Cross-validation, ensemble and class-weight tuning

Out-of-fold (OOF) probabilities from the top-K models give a more reliable estimate than a single holdout, and let us fit per-class multipliers that push the predictions toward an even spread (this helps *balanced* accuracy). Because the multipliers are tuned on the same OOF predictions they are evaluated on, treat the post-tuning number as slightly optimistic.

In [ ]:
def tune_class_weights(P, yy, rounds=2, grid=np.linspace(0.6, 1.6, 21)):
    w = np.ones(P.shape[1])
    best = balanced_accuracy_score(yy, (P * w).argmax(1))
    for _ in range(rounds):
        for c in range(P.shape[1]):
            keep = w[c]
            for g in grid:
                w[c] = g
                s = balanced_accuracy_score(yy, (P * w).argmax(1))
                if s > best + 1e-9:
                    best, keep = s, g
            w[c] = keep
    return w, best

oof, class_w = {}, np.ones(n_cls)
if RUN_CV:
    skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)
    for name in top_names:
        spec, X = MODELS[name], FEATS[MODELS[name]["feats"]][:n_train]
        O = np.zeros((n_train, n_cls)); t0 = time.time()
        for f, (a, b) in enumerate(skf.split(X, y)):
            O[b] = get_scores(fit_model(spec, X[a], y[a]), X[b])
        oof[name] = O
        print(f"{name:22s} CV bal_acc={balanced_accuracy_score(y, O.argmax(1)):.4f}  ({time.time()-t0:.0f}s)")

    ens_oof = np.mean(list(oof.values()), axis=0)
    print("Ensemble CV bal_acc:", round(balanced_accuracy_score(y, ens_oof.argmax(1)), 4))
    class_w, tuned = tune_class_weights(ens_oof, y)
    print("After class-weight tuning (optimistic):", round(tuned, 4))

In [ ]:
if RUN_CV:
    pred = (ens_oof * class_w).argmax(1)
    recall = pd.Series(confusion_matrix(y, pred, normalize="true").diagonal(), index=CLASSES).sort_values()
    print("Per-class recall (weakest first):"); print(recall.round(3).to_string())

    fig, ax = plt.subplots(figsize=(10, 9))
    ConfusionMatrixDisplay.from_predictions(y, pred, display_labels=CLASSES, normalize="true",
                                            xticks_rotation=90, values_format=".2f", ax=ax, colorbar=False)
    ax.set_title("OOF confusion matrix (row-normalised)"); plt.tight_layout(); plt.show()

## 5. Final fit and submission

Fits the selected models on **all** training rows and predicts the test set. The submission layout is copied from `sample_submission` when one exists; otherwise a simple `id` + label file is written. **Compare it with the competition's Data tab before submitting**: if the metric needs probabilities or a different column name, adjust the last cell.

In [ ]:
test_P = []
for name in top_names:
    spec = MODELS[name]
    m = fit_model(spec, FEATS[spec["feats"]][:n_train], y)
    test_P.append(get_scores(m, FEATS[spec["feats"]][n_train:]))
    print("fitted", name)

P_test = np.mean(test_P, axis=0) * class_w
P_test = P_test / P_test.sum(1, keepdims=True)
test_pred = CLASSES[P_test.argmax(1)]
pd.Series(test_pred).value_counts().head(n_cls)

In [ ]:
ids = test[ID_COL] if ID_COL else pd.Series(np.arange(len(test)), name="id")

if sub_path:
    sub = read_any(sub_path).iloc[:0].copy()
    cols = list(sub.columns)
    if set(map(str, CLASSES)) <= set(cols):                       # probability-style submission
        sub = pd.DataFrame(P_test, columns=list(map(str, CLASSES)))
        sub.insert(0, cols[0], ids.values)
    else:                                                          # label-style submission
        sub = pd.DataFrame({cols[0]: ids.values, cols[1]: test_pred})
else:
    sub = pd.DataFrame({ID_COL or "id": ids.values, LABEL_COL: test_pred})

sub.to_parquet("submission.parquet", index=False)
print(sub.shape); sub.head()